In [1]:
%pip install datasets tokenizers

In [32]:
import math
import torch
import torch.nn as nn

from datasets import load_dataset
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

In [33]:
# ============================================================
# 1. Global Hyperparameters and Hardware Configuration
# ============================================================

VOCAB_SIZE = 4000
MAX_SEQ_LEN = 128

# Model architecture configuration (~13.7M parameters)
D_MODEL = 384
NUM_HEADS = 6
NUM_LAYERS = 6
D_FF = 1536
DROPOUT = 0.2

BATCH_SIZE = 32
LEARNING_RATE = 3e-4
EPOCHS = 30

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Execution Device:", device)

Execution Device: cuda


### Model Regularization
We increase the dropout rate to `0.2` and use weight decay to reduce overfitting and improve generalizability during the training process.

In [31]:
# Initialize hyperparameters with increased dropout for regularization
DROPOUT = 0.2

# Instantiate the custom MiniGPT model with updated dropout
model = MiniGPT(
    vocab_size=VOCAB_SIZE,
    max_seq_len=MAX_SEQ_LEN,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    d_ff=D_FF,
    dropout=DROPOUT
)
model = model.to(device)
print("Model successfully instantiated with DROPOUT =", DROPOUT)

Model successfully instantiated with DROPOUT = 0.2


In [7]:
import math
import torch
import torch.nn as nn

# ============================================================
# DEFINITION COMPLETE DE MINIGPT (RE-EXECUTION)
# ============================================================

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads, max_seq_len, dropout):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.Wq = nn.Linear(d_model, d_model, bias=False)
        self.Wk = nn.Linear(d_model, d_model, bias=False)
        self.Wv = nn.Linear(d_model, d_model, bias=False)
        self.Wo = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len))
        self.register_buffer("mask", mask)

    def forward(self, x):
        B, T, C = x.shape
        Q = self.Wq(x).reshape(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.Wk(x).reshape(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.Wv(x).reshape(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        current_mask = self.mask[:T, :T]
        scores = scores.masked_fill(current_mask == 0, float("-inf"))

        attention_weights = torch.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)

        output = attention_weights @ V
        output = output.transpose(1, 2).contiguous().reshape(B, T, self.d_model)
        return self.Wo(output)

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )
    def forward(self, x):
        return self.network(x)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, max_seq_len, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attention = CausalSelfAttention(d_model, num_heads, max_seq_len, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        x = x + self.feed_forward(self.norm2(x))
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, max_seq_len, d_model, num_heads, num_layers, d_ff, dropout):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_seq_len, d_model)
        self.dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff, max_seq_len, dropout)
            for _ in range(num_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, token_ids):
        B, T = token_ids.shape
        token_embeddings = self.token_embedding(token_ids)
        positions = torch.arange(T, device=token_ids.device)
        position_embeddings = self.position_embedding(positions)
        x = self.dropout(token_embeddings + position_embeddings)
        for block in self.blocks:
            x = block(x)
        x = self.final_norm(x)
        return self.lm_head(x)

# ============================================================
# RECREATION INSTANTANEE DU MODELE
# ============================================================
DROPOUT = 0.2
model = MiniGPT(
    vocab_size=VOCAB_SIZE,
    max_seq_len=MAX_SEQ_LEN,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    d_ff=D_FF,
    dropout=DROPOUT
)
model = model.to(device)
print("MiniGPT est maintenant défini et instancié avec DROPOUT =", DROPOUT)

MiniGPT est maintenant défini et instancié avec DROPOUT = 0.2


In [34]:
# ============================================================
# 2. Dataset Loading
# ============================================================

dataset = load_dataset(
    "Jim-Webfox/HomerSimpson",
    split="train"
)
texts = dataset["text"]
print("Total dialogues processed:", len(texts))

Total dialogues processed: 15714


In [35]:
# ============================================================
# 3. Text Normalization and Token Mapping
# ============================================================

def clean_text(text):
    text = text.replace("### Human:", "<HUMAN>")
    text = text.replace("### Assistant:", "<ASSISTANT>")
    return text

texts = [clean_text(text) for text in texts]

### Data Filtering and Noise Reduction
To improve output quality, we apply a regex-based filter to exclude noisy dialogues containing long numeric sequences (such as timestamps or IDs) and short sentences.

In [36]:
import re

def is_clean_enough(text):
    # Exclude entries containing long numeric sequences (such as serials or timestamps)
    if re.search(r'\d{3,}', text):
        return False
    # Exclude extremely short dialogue turns
    if len(text.strip()) < 10:
        return False
    return True

texts_clean = [t for t in texts if is_clean_enough(t)]
print(f"Raw dialogues: {len(texts)}")
print(f"Filtered dialogues: {len(texts_clean)}")
texts = texts_clean

Raw dialogues: 15714
Filtered dialogues: 15608


In [37]:
# ============================================================
# 4. Byte-Pair Encoding (BPE) Tokenizer Training
# ============================================================

special_tokens = [
    "<PAD>",
    "<UNK>",
    "<BOS>",
    "<EOS>",
    "<HUMAN>",
    "<ASSISTANT>"
]

tokenizer = Tokenizer(models.BPE(unk_token="<UNK>"))
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=special_tokens
)

tokenizer.train_from_iterator(texts, trainer=trainer)
VOCAB_SIZE = tokenizer.get_vocab_size()
print("Final Vocabulary Size:", VOCAB_SIZE)

Final Vocabulary Size: 4000


In [38]:
# Map special token identifiers
PAD_ID = tokenizer.token_to_id("<PAD>")
UNK_ID = tokenizer.token_to_id("<UNK>")
BOS_ID = tokenizer.token_to_id("<BOS>")
EOS_ID = tokenizer.token_to_id("<EOS>")
HUMAN_ID = tokenizer.token_to_id("<HUMAN>")
ASSISTANT_ID = tokenizer.token_to_id("<ASSISTANT>")

In [39]:
# ============================================================
# 5. PyTorch Dataset Implementation
# ============================================================

class HomerDataset(torch.utils.data.Dataset):
    def __init__(self, texts, tokenizer, max_seq_len):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        text = self.texts[index]
        ids = self.tokenizer.encode(text).ids

        # Prepend BOS and append EOS markers
        ids = [BOS_ID] + ids + [EOS_ID]

        # Clamp slice to accommodate the +1 target offset
        ids = ids[:self.max_seq_len + 1]

        # Apply padding up to target length
        while len(ids) < (self.max_seq_len + 1):
            ids.append(PAD_ID)

        ids = torch.tensor(ids, dtype=torch.long)

        # Causal shift split for autoregressive training
        x = ids[:-1]
        y = ids[1:]
        return x, y

In [14]:
# ============================================================
# 8. CAUSAL SELF ATTENTION
# ============================================================

class CausalSelfAttention(nn.Module):

    def __init__(
        self,
        d_model,
        num_heads,
        max_seq_len,
        dropout
    ):

        super().__init__()


        assert (
            d_model % num_heads == 0
        )


        self.d_model = d_model

        self.num_heads = num_heads

        self.head_dim = (
            d_model // num_heads
        )


        # ========================================
        # Q K V
        # ========================================

        self.Wq = nn.Linear(
            d_model,
            d_model,
            bias=False
        )


        self.Wk = nn.Linear(
            d_model,
            d_model,
            bias=False
        )


        self.Wv = nn.Linear(
            d_model,
            d_model,
            bias=False
        )


        # Projection après les heads

        self.Wo = nn.Linear(
            d_model,
            d_model
        )


        self.dropout = nn.Dropout(
            dropout
        )


        # ========================================
        # CAUSAL MASK
        # ========================================

        mask = torch.tril(

            torch.ones(
                max_seq_len,
                max_seq_len
            )
        )


        self.register_buffer(
            "mask",
            mask
        )


    def forward(self, x):

        B, T, C = x.shape


        # ========================================
        # Q = XWq
        # K = XWk
        # V = XWv
        # ========================================

        Q = self.Wq(x)

        K = self.Wk(x)

        V = self.Wv(x)


        # ========================================
        # MULTI HEAD
        #
        # (B,T,128)
        #
        # ->
        #
        # (B,T,4,32)
        # ========================================

        Q = Q.reshape(

            B,
            T,
            self.num_heads,
            self.head_dim
        )


        K = K.reshape(

            B,
            T,
            self.num_heads,
            self.head_dim
        )


        V = V.reshape(

            B,
            T,
            self.num_heads,
            self.head_dim
        )


        # ----------------------------------------
        # (B, heads, T, head_dim)
        # ----------------------------------------

        Q = Q.transpose(1, 2)

        K = K.transpose(1, 2)

        V = V.transpose(1, 2)


        # ========================================
        # ATTENTION SCORES
        #
        # Q K^T
        # ========================================

        scores = (
            Q
            @ K.transpose(-2, -1)
        )


        # ========================================
        # SCALE
        # ========================================

        scores = (

            scores
            / math.sqrt(
                self.head_dim
            )

        )


        # ========================================
        # CAUSAL MASK
        # ========================================

        current_mask = self.mask[
            :T,
            :T
        ]


        scores = scores.masked_fill(

            current_mask == 0,

            float("-inf")
        )


        # ========================================
        # SOFTMAX
        # ========================================

        attention_weights = (
            torch.softmax(
                scores,
                dim=-1
            )
        )


        attention_weights = (
            self.dropout(
                attention_weights
            )
        )


        # ========================================
        # Attention × V
        # ========================================

        output = (
            attention_weights
            @ V
        )


        # ========================================
        # REASSEMBLER LES HEADS
        # ========================================

        output = (
            output
            .transpose(1, 2)
            .contiguous()
        )


        output = output.reshape(

            B,
            T,
            self.d_model
        )


        output = self.Wo(
            output
        )


        return output

In [15]:
# ============================================================
# 9. FEED FORWARD
# ============================================================

class FeedForward(nn.Module):

    def __init__(
        self,
        d_model,
        d_ff,
        dropout
    ):

        super().__init__()


        self.network = nn.Sequential(

            nn.Linear(
                d_model,
                d_ff
            ),

            nn.GELU(),

            nn.Linear(
                d_ff,
                d_model
            ),

            nn.Dropout(
                dropout
            )
        )


    def forward(self, x):

        return self.network(x)

In [16]:
# ============================================================
# 10. TRANSFORMER BLOCK
# ============================================================

class TransformerBlock(nn.Module):

    def __init__(
        self,
        d_model,
        num_heads,
        d_ff,
        max_seq_len,
        dropout
    ):

        super().__init__()


        self.norm1 = nn.LayerNorm(
            d_model
        )


        self.attention = (
            CausalSelfAttention(

                d_model,

                num_heads,

                max_seq_len,

                dropout
            )
        )


        self.norm2 = nn.LayerNorm(
            d_model
        )


        self.feed_forward = FeedForward(

            d_model,

            d_ff,

            dropout
        )


    def forward(self, x):

        # Attention + residual

        x = (

            x
            +
            self.attention(
                self.norm1(x)
            )

        )


        # Feed Forward + residual

        x = (

            x
            +
            self.feed_forward(
                self.norm2(x)
            )

        )


        return x

In [17]:
# ============================================================
# 11. MINI GPT
# ============================================================

class MiniGPT(nn.Module):

    def __init__(
        self,
        vocab_size,
        max_seq_len,
        d_model,
        num_heads,
        num_layers,
        d_ff,
        dropout
    ):

        super().__init__()


        self.max_seq_len = (
            max_seq_len
        )


        # ========================================
        # TOKEN EMBEDDINGS
        # ========================================

        self.token_embedding = (
            nn.Embedding(

                vocab_size,

                d_model
            )
        )


        # ========================================
        # POSITION EMBEDDINGS
        # ========================================

        self.position_embedding = (
            nn.Embedding(

                max_seq_len,

                d_model
            )
        )


        self.dropout = nn.Dropout(
            dropout
        )


        # ========================================
        # TRANSFORMER BLOCKS
        # ========================================

        self.blocks = nn.ModuleList([

            TransformerBlock(

                d_model,

                num_heads,

                d_ff,

                max_seq_len,

                dropout
            )

            for _ in range(
                num_layers
            )

        ])


        self.final_norm = (
            nn.LayerNorm(
                d_model
            )
        )


        # ========================================
        # OUTPUT
        #
        # 128 -> 4000
        # ========================================

        self.lm_head = nn.Linear(

            d_model,

            vocab_size,

            bias=False
        )


    def forward(self, token_ids):

        B, T = token_ids.shape


        # ========================================
        # TOKEN EMBEDDINGS
        # ========================================

        token_embeddings = (
            self.token_embedding(
                token_ids
            )
        )


        # ========================================
        # POSITIONS
        # ========================================

        positions = torch.arange(

            T,

            device=token_ids.device
        )


        position_embeddings = (
            self.position_embedding(
                positions
            )
        )


        # ========================================
        # TOKEN + POSITION
        # ========================================

        x = (

            token_embeddings
            +
            position_embeddings

        )


        x = self.dropout(x)


        # ========================================
        # TRANSFORMER
        # ========================================

        for block in self.blocks:

            x = block(x)


        x = self.final_norm(x)


        # ========================================
        # LOGITS
        # ========================================

        logits = self.lm_head(x)


        return logits

In [ ]:
# ============================================================
# 12. CREATION DU MODELE
# ============================================================

model = MiniGPT(

    vocab_size=VOCAB_SIZE,

    max_seq_len=MAX_SEQ_LEN,

    d_model=D_MODEL,

    num_heads=NUM_HEADS,

    num_layers=NUM_LAYERS,

    d_ff=D_FF,

    dropout=DROPOUT
)


model = model.to(device)

In [18]:
# ============================================================
# 13. NOMBRE DE PARAMETRES
# ============================================================

num_parameters = sum(

    parameter.numel()

    for parameter
    in model.parameters()

)


print(
    f"\nNumber of parameters: "
    f"{num_parameters:,}"
)


Number of parameters: 13,761,792


In [19]:
# ============================================================
# 14. TRAIN / VALIDATION SPLIT
# ============================================================

full_dataset = HomerDataset(

    texts,

    tokenizer,

    MAX_SEQ_LEN
)


train_size = int(

    0.9
    *
    len(full_dataset)

)


validation_size = (

    len(full_dataset)
    -
    train_size

)


train_dataset, validation_dataset = (

    torch.utils.data.random_split(

        full_dataset,

        [
            train_size,
            validation_size
        ],

        generator=torch.Generator()
        .manual_seed(42)
    )

)

In [20]:
# ============================================================
# 15. DATALOADERS
# ============================================================

train_loader = (

    torch.utils.data.DataLoader(

        train_dataset,

        batch_size=BATCH_SIZE,

        shuffle=True
    )

)


validation_loader = (

    torch.utils.data.DataLoader(

        validation_dataset,

        batch_size=BATCH_SIZE,

        shuffle=False
    )

)

In [21]:
# ============================================================
# 16. LOSS
# ============================================================

criterion = nn.CrossEntropyLoss(

    ignore_index=PAD_ID
)

In [22]:
# ============================================================
# 17. OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=LEARNING_RATE
)

### Optimization Configuration
We use the AdamW optimizer coupled with L2 regularization (`weight_decay=0.01`) to stabilize the learning of weights and avoid overparameterization issues.

In [23]:
# Optimiseur AdamW avec pénalité sur les grands poids (Weight Decay)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.01
)
print("Optimiseur AdamW configuré avec weight_decay=0.01")

Optimiseur AdamW configuré avec weight_decay=0.01


### Cosine Annealing Learning Rate Scheduler
To achieve stable convergence for the 13.7M parameters model, we construct a custom Cosine Learning Rate Scheduler with a linear warmup phase.

In [40]:
from torch.optim.lr_scheduler import LambdaLR

def get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps, num_cycles=0.5):
    def lr_lambda(current_step):
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        progress = float(current_step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * float(num_cycles) * 2.0 * progress)))
    return LambdaLR(optimizer, lr_lambda)

# Calculate overall optimization parameters
num_training_steps = len(train_loader) * EPOCHS
num_warmup_steps = int(0.1 * num_training_steps)

# Initialize learning rate scheduler
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps)
print(f"Scheduler configured with {num_warmup_steps} warmup steps out of {num_training_steps} total steps.")

Scheduler configured with 1317 warmup steps out of 13170 total steps.


In [25]:
# ============================================================
# 18. VALIDATION
# ============================================================

def evaluate():

    model.eval()


    total_loss = 0


    with torch.no_grad():

        for x, y in validation_loader:

            x = x.to(device)

            y = y.to(device)


            logits = model(x)


            loss = criterion(

                logits.reshape(
                    -1,
                    VOCAB_SIZE
                ),

                y.reshape(-1)
            )


            total_loss += loss.item()


    return (

        total_loss
        /
        len(validation_loader)

    )

In [41]:
# ============================================================
# 6. Training Loop (Baseline Architecture Evaluation)
# ============================================================

PATIENCE = 3
best_validation_loss = float('inf')
patience_counter = 0
best_model_state = None

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(
            logits.reshape(-1, VOCAB_SIZE),
            y.reshape(-1)
        )
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()

    train_loss = total_train_loss / len(train_loader)
    validation_loss = evaluate()

    print(
        f"Epoch {epoch + 1}/{EPOCHS}"
        f" | Train Loss: {train_loss:.4f}"
        f" | Validation Loss: {validation_loss:.4f}"
    )

    # Early stopping criteria evaluation
    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        patience_counter = 0
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print("   => Checkpoint saved (improved validation loss)")
    else:
        patience_counter += 1
        print(f"   => No improvement. Patience: {patience_counter}/{PATIENCE}")

    if patience_counter >= PATIENCE:
        print(f"\n[Early Stopping] Triggered at epoch {epoch + 1} due to loss stagnation.")
        break

if best_model_state is not None:
    model.load_state_dict({k: v.to(device) for k, v in best_model_state.items()})
    print("Optimal parameters successfully restored to memory.")

Epoch 1/30 | Train Loss: 8.4373 | Validation Loss: 8.4359
   => Checkpoint saved (improved validation loss)
Epoch 2/30 | Train Loss: 8.4374 | Validation Loss: 8.4359
   => No improvement. Patience: 1/3
Epoch 3/30 | Train Loss: 8.4366 | Validation Loss: 8.4359
   => No improvement. Patience: 2/3
Epoch 4/30 | Train Loss: 8.4380 | Validation Loss: 8.4359
   => No improvement. Patience: 3/3

[Early Stopping] Triggered at epoch 4 due to loss stagnation.
Optimal parameters successfully restored to memory.


### Training Execution with Early Stopping and Gradient Clipping
We run the optimization loop incorporating gradient clipping to prevent exploding gradients and a validation loss tracker to stop training when convergence is reached.

In [42]:
# ============================================================
# 7. Final Training Loop with Gradient Clipping and LR Warmup
# ============================================================

PATIENCE = 3
best_validation_loss = float('inf')
patience_counter = 0
best_model_state = None

scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps)

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits.reshape(-1, VOCAB_SIZE), y.reshape(-1))
        loss.backward()

        # Gradient norm clipping to preserve training stability
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        total_train_loss += loss.item()

    train_loss = total_train_loss / len(train_loader)
    validation_loss = evaluate()
    current_lr = scheduler.get_last_lr()[0]

    print(
        f"Epoch {epoch + 1}/{EPOCHS}"
        f" | Train Loss: {train_loss:.4f}"
        f" | Validation Loss: {validation_loss:.4f}"
        f" | LR: {current_lr:.6f}"
    )

    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        patience_counter = 0
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print("   => Checkpoint saved (improved validation loss)")
    else:
        patience_counter += 1
        print(f"   => No improvement. Patience: {patience_counter}/{PATIENCE}")

    if patience_counter >= PATIENCE:
        print(f"\n[Early Stopping] Triggered at epoch {epoch + 1}.")
        break

if best_model_state is not None:
    model.load_state_dict({k: v.to(device) for k, v in best_model_state.items()})
    print("Optimal parameters successfully restored to memory.")

Epoch 1/30 | Train Loss: 8.4381 | Validation Loss: 8.4359 | LR: 0.000100
   => Checkpoint saved (improved validation loss)
Epoch 2/30 | Train Loss: 8.4376 | Validation Loss: 8.4359 | LR: 0.000200
   => No improvement. Patience: 1/3
Epoch 3/30 | Train Loss: 8.4382 | Validation Loss: 8.4359 | LR: 0.000300
   => No improvement. Patience: 2/3
Epoch 4/30 | Train Loss: 8.4373 | Validation Loss: 8.4359 | LR: 0.000299
   => No improvement. Patience: 3/3

[Early Stopping] Triggered at epoch 4.
Optimal parameters successfully restored to memory.


In [43]:
# ============================================================
# 8. Inference with Temperature Scaling and Top-K Filtering
# ============================================================

def generate(
    prompt,
    max_new_tokens=50,
    temperature=0.6,
    top_k=40
):
    model.eval()
    text = f"<HUMAN> {prompt}\n<ASSISTANT> "
    ids = tokenizer.encode(text).ids
    ids = [BOS_ID] + ids

    tokens = torch.tensor([ids], dtype=torch.long, device=device)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            model_input = tokens[:, -MAX_SEQ_LEN:]
            logits = model(model_input)
            logits = logits[:, -1, :] / temperature

            if top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')

            probabilities = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probabilities, num_samples=1)
            tokens = torch.cat([tokens, next_token], dim=1)

            if next_token.item() == EOS_ID:
                break

    generated_ids = tokens[0].tolist()
    return tokenizer.decode(generated_ids)

In [44]:
# ============================================================
# 9. Model Inference Evaluation
# ============================================================

print(generate("Homer, what do you think about donuts?"))

 Homer, what do you think about donuts? vycc R might exciting place scared card hum Think She Edison pounds go met cashents m Over children projectnowumber angel getting rules Six Why another ser� together animalfield beans Giveittle parent town sand cloth name Patty WIT sistereg fab welcome


### Interactive Inference Interface
The section below implements the evaluation loop using Top-K sampling and scaled temperature configurations for inference.

In [45]:
# ============================================================
# 10. Interactive Chat Interface
# ============================================================

def generate_homer_reply(prompt, max_new_tokens=50, temperature=0.6, top_k=40):
    model.eval()
    text = f"<HUMAN> {prompt}\n<ASSISTANT> "

    ids = tokenizer.encode(text).ids
    ids = [BOS_ID] + ids
    prompt_len = len(ids)

    tokens = torch.tensor([ids], dtype=torch.long, device=device)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            model_input = tokens[:, -MAX_SEQ_LEN:]
            logits = model(model_input)
            logits = logits[:, -1, :] / temperature

            if top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')

            probabilities = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probabilities, num_samples=1)
            tokens = torch.cat([tokens, next_token], dim=1)

            if next_token.item() == EOS_ID:
                break

    generated_tokens = tokens[0][prompt_len:].tolist()
    return tokenizer.decode(generated_tokens).strip()

def start_chat_with_homer():
    print("--- Chat initialized. Type 'exit' to quit. ---\n")
    while True:
        user_input = input("You: ")
        if user_input.strip().lower() in ['exit', 'quit']:
            print("Homer: Bye! D'oh!")
            break
        if not user_input.strip():
            continue

        reply = generate_homer_reply(user_input, max_new_tokens=50, temperature=0.6, top_k=40)
        print(f"Homer: {reply}\n")

# Run interface
start_chat_with_homer()

--- Chat initialized. Type 'exit' to quit. ---

You: exit
Homer: Bye! D'oh!
